# Improved Event Chain Model for Narrative Similarity

## Key Improvements:
1. Semantic embeddings instead of string matching
2. Theme extraction and comparison
3. Enhanced event extraction (emotional valence, turning points)
4. Better outcome comparison
5. Tunable weights for different components
6. Optimal event alignment using Hungarian algorithm

In [1]:
import os
import json
import time
from typing import List, Dict, Any, Tuple
from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from sentence_transformers import SentenceTransformer, util

import google.generativeai as genai

print("Imports complete.")

/home/saha/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports complete.


In [3]:
# Configuration
API_KEY = os.getenv("GEMINI_API_KEY")
if not API_KEY or API_KEY == "YOUR_API_KEY_HERE":
    raise ValueError("Please set GEMINI_API_KEY env var or replace YOUR_API_KEY_HERE with your actual key.")

genai.configure(api_key=API_KEY)

# Model configuration
GEMINI_MODEL_NAME = "gemini-2.0-flash"
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"  #

# Data paths
DEV_PATH = "../Data/SemEval2026-Task_4-dev-v1/dev_track_a.jsonl"
SAMPLE_PATH = "../Data/SemEval2026-Task_4-sample-v1/sample_track_a.jsonl"

print("Config ready.")
print("Using Gemini model:", GEMINI_MODEL_NAME)
print("Using embedding model:", EMBEDDING_MODEL_NAME)
print("Dev file path:", DEV_PATH)
print("Sample file path:", SAMPLE_PATH)

Config ready.
Using Gemini model: gemini-2.0-flash
Using embedding model: all-MiniLM-L6-v2
Dev file path: ../Data/SemEval2026-Task_4-dev-v1/dev_track_a.jsonl
Sample file path: ../Data/SemEval2026-Task_4-sample-v1/sample_track_a.jsonl


In [4]:
# Initialize models
print("Loading embedding model...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print("Embedding model loaded.")

print("Initializing Gemini model...")
event_model = genai.GenerativeModel(GEMINI_MODEL_NAME)
print("Gemini model initialized.")

Loading embedding model...
Embedding model loaded.
Initializing Gemini model...
Gemini model initialized.


In [5]:
# Load data
df_dev = pd.read_json(DEV_PATH, lines=True)
print("Loaded dev set:", df_dev.shape)
print(df_dev.head(2))

Loaded dev set: (200, 4)
                                         anchor_text  \
0  The book follows an international organization...   
1  Glenn Tyler (Elvis Presley), a childish 25-yea...   

                                              text_a  \
0  The old grandmother Tina arrives in town to at...   
1  Bill Babbitt supported the death penalty, unti...   

                                              text_b  text_a_is_closer  
0  The nano-plague that poisoned Earth's water su...             False  
1  A white-collar suburban father Kyle (Fran Kran...              True  


In [6]:
# Semantic similarity function
def semantic_similarity(text1: str, text2: str) -> float:
    """
    Compute cosine similarity between two texts using embeddings.
    Returns value between 0 and 1.
    """
    if not text1 or not text2:
        return 0.0
    
    if not text1.strip() or not text2.strip():
        return 0.0
    
    emb1 = embedding_model.encode(text1, convert_to_tensor=True)
    emb2 = embedding_model.encode(text2, convert_to_tensor=True)
    
    similarity = util.cos_sim(emb1, emb2)
    return float(similarity.item())

# Test
test_sim = semantic_similarity("lost item", "misplaced object")
print(f"Test similarity ('lost item' vs 'misplaced object'): {test_sim:.3f}")
print("Semantic similarity function ready.")

Test similarity ('lost item' vs 'misplaced object'): 0.577
Semantic similarity function ready.


In [7]:
# Enhanced event extraction prompt
EVENT_EXTRACTION_PROMPT = """
You MUST output ONLY valid JSON.
No explanations. No comments. No text outside the JSON object.

Extract the THEME and EVENT CHAIN from the story.

Return EXACTLY ONE JSON OBJECT in this format:

{{
  "theme": "the central idea, moral, or concept of the story (1-2 sentences)",
  "outcome_description": "brief description of how the story ends and what the final state is (1-2 sentences)",
  "events": [
    {{
      "index": 1,
      "actors": ["actor1", "actor2"],
      "action": "main action verb phrase",
      "object": "object of action",
      "result": "what changed after the event",
      "emotional_valence": "positive/negative/neutral",
      "is_turning_point": true,
      "is_outcome_event": false
    }}
  ]
}}

Guidelines:
- Extract 3-7 key events (not every detail)
- Focus on events that drive the narrative forward
- Mark turning points (events that change the story direction)
- Mark outcome events (events near the end that show final state)
- emotional_valence: positive (good outcome), negative (bad outcome), neutral

STORY:
{story}
"""

print("Enhanced extraction prompt ready.")

Enhanced extraction prompt ready.


In [8]:
# Event extraction function
def extract_narrative_structure(
    story: str,
    max_retries: int = 3,
    sleep_sec: float = 0.5,
    print_raw_on_failure: bool = True
) -> Dict[str, Any]:
    """
    Use Gemini to extract theme, events, and outcome description.
    Returns: {
        "theme": str,
        "outcome_description": str,
        "events": [list of event dicts]
    }
    """
    last_raw = None

    for attempt in range(max_retries):
        try:
            prompt = EVENT_EXTRACTION_PROMPT.format(story=story)
            response = event_model.generate_content(prompt)
            text = response.text.strip()
            last_raw = text

            # Strip code fences
            if text.startswith("```"):
                text = text.strip("`").strip()
                if text.lower().startswith("json"):
                    text = text[4:].strip()

            # Try direct parse
            try:
                data = json.loads(text)
            except json.JSONDecodeError:
                # Extract JSON from text
                first_brace = text.find("{")
                last_brace = text.rfind("}")
                if first_brace != -1 and last_brace != -1 and last_brace > first_brace:
                    candidate = text[first_brace:last_brace+1]
                    data = json.loads(candidate)
                else:
                    raise

            # Validate and normalize
            theme = str(data.get("theme", "")).strip()
            outcome_desc = str(data.get("outcome_description", "")).strip()
            events = data.get("events", [])
            
            if not isinstance(events, list):
                raise ValueError("'events' is not a list")

            normalized_events = []
            for idx, ev in enumerate(events):
                normalized_events.append({
                    "index": int(ev.get("index", idx + 1)),
                    "actors": [str(a).strip() for a in ev.get("actors", []) if str(a).strip()],
                    "action": str(ev.get("action", "")).strip(),
                    "object": str(ev.get("object", "")).strip(),
                    "result": str(ev.get("result", "")).strip(),
                    "emotional_valence": str(ev.get("emotional_valence", "neutral")).strip().lower(),
                    "is_turning_point": bool(ev.get("is_turning_point", False)),
                    "is_outcome_event": bool(ev.get("is_outcome_event", False)),
                })

            return {
                "theme": theme,
                "outcome_description": outcome_desc,
                "events": sorted(normalized_events, key=lambda e: e["index"])
            }

        except Exception as e:
            print(f"[extract_narrative_structure] Attempt {attempt+1} failed:", repr(e))
            time.sleep(sleep_sec)

    # Fallback
    if print_raw_on_failure and last_raw is not None:
        print("\n--- RAW GEMINI OUTPUT (last attempt) ---")
        print(last_raw)
        print("--- END RAW OUTPUT ---\n")

    print("[extract_narrative_structure] Falling back to empty structure.")
    return {
        "theme": "",
        "outcome_description": "",
        "events": []
    }

print("Event extraction function ready.")

Event extraction function ready.


In [25]:
# Event-level similarity using embeddings
def event_similarity(ev_a: Dict[str, Any], ev_b: Dict[str, Any]) -> float:
    """
    Compute semantic similarity between two events.
    Weighted average of: actors, action, object, result.
    """
    weights = {
        'action': 0.40,
        'actors': 0.25,
        'object': 0.20,
        'result': 0.15
    }
    
    scores = {}
    
    # Action similarity
    scores['action'] = semantic_similarity(
        ev_a.get('action', ''),
        ev_b.get('action', '')
    )
    
    # Actors similarity (join list into string)
    actors_a = ', '.join(ev_a.get('actors', []))
    actors_b = ', '.join(ev_b.get('actors', []))
    scores['actors'] = semantic_similarity(actors_a, actors_b)
    
    # Object similarity
    scores['object'] = semantic_similarity(
        ev_a.get('object', ''),
        ev_b.get('object', '')
    )
    
    # Result similarity
    scores['result'] = semantic_similarity(
        ev_a.get('result', ''),
        ev_b.get('result', '')
    )
    
    # Weighted average
    total = sum(weights[k] * scores[k] for k in weights.keys())
    return float(total)

print("Event similarity function ready.")

import difflib

def hybrid_event_similarity(ev_a: Dict[str, Any], ev_b: Dict[str, Any]) -> float:
    """
    Hybrid: 70% semantic + 30% exact string matching
    """
    weights = {
        'action': 0.40,
        'actors': 0.25,
        'object': 0.20,
        'result': 0.15
    }
    
    def hybrid_field_similarity(text1: str, text2: str) -> float:
        if not text1 or not text2:
            return 0.0
        
        # Semantic similarity
        sem_sim = semantic_similarity(text1, text2)
        
        # Exact string similarity
        exact_sim = difflib.SequenceMatcher(None, text1.lower(), text2.lower()).ratio()
        
        # Combine: favor semantic but reward exact matches
        return 0.70 * sem_sim + 0.30 * exact_sim
    
    scores = {}
    scores['action'] = hybrid_field_similarity(ev_a.get('action', ''), ev_b.get('action', ''))
    
    actors_a = ', '.join(ev_a.get('actors', []))
    actors_b = ', '.join(ev_b.get('actors', []))
    scores['actors'] = hybrid_field_similarity(actors_a, actors_b)
    
    scores['object'] = hybrid_field_similarity(ev_a.get('object', ''), ev_b.get('object', ''))
    scores['result'] = hybrid_field_similarity(ev_a.get('result', ''), ev_b.get('result', ''))
    
    return float(sum(weights[k] * scores[k] for k in weights.keys()))

event_similarity = hybrid_event_similarity
print("Hybrid event similarity ready.")

Event similarity function ready.
Hybrid event similarity ready.


In [26]:

def align_event_chains_optimal(
    chain_a: List[Dict[str, Any]],
    chain_b: List[Dict[str, Any]]
) -> Tuple[List[Tuple[Dict, Dict]], float]:
    """
    Optimally align two event chains using Hungarian algorithm.
    Returns: (aligned_pairs, avg_similarity)
    """
    if not chain_a or not chain_b:
        return [], 0.0
    
    n_a = len(chain_a)
    n_b = len(chain_b)
    
    # Create similarity matrix
    sim_matrix = np.zeros((n_a, n_b))
    for i, ev_a in enumerate(chain_a):
        for j, ev_b in enumerate(chain_b):
            sim_matrix[i, j] = event_similarity(ev_a, ev_b)
    
    # Hungarian algorithm (maximize by negating)
    row_ind, col_ind = linear_sum_assignment(-sim_matrix)
    
    # Build aligned pairs
    aligned_pairs = [
        (chain_a[i], chain_b[j]) 
        for i, j in zip(row_ind, col_ind)
    ]
    
    # Average similarity of matched pairs
    if len(row_ind) > 0:
        avg_sim = sim_matrix[row_ind, col_ind].mean()
    else:
        avg_sim = 0.0
    
    return aligned_pairs, float(avg_sim)

print("Optimal alignment function ready.")

Optimal alignment function ready.


In [11]:
# Sequence order preservation using LCS
def sequence_order_score(aligned_pairs: List[Tuple[Dict, Dict]]) -> float:
    """
    Measure how well the sequence order is preserved.
    Uses LCS ratio of indices.
    """
    if not aligned_pairs:
        return 1.0
    
    indices_a = [pair[0].get('index', 0) for pair in aligned_pairs]
    indices_b = [pair[1].get('index', 0) for pair in aligned_pairs]
    
    # Simple LCS length
    def lcs_length(seq1, seq2):
        m, n = len(seq1), len(seq2)
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if seq1[i-1] == seq2[j-1]:
                    dp[i][j] = dp[i-1][j-1] + 1
                else:
                    dp[i][j] = max(dp[i-1][j], dp[i][j-1])
        
        return dp[m][n]
    
    lcs_len = lcs_length(indices_a, indices_b)
    max_len = max(len(indices_a), len(indices_b))
    
    return lcs_len / max_len if max_len > 0 else 1.0

print("Sequence order function ready.")

Sequence order function ready.


In [12]:
# Enhanced outcome similarity
def outcome_similarity(
    struct_a: Dict[str, Any],
    struct_b: Dict[str, Any]
) -> float:
    """
    Compare outcomes using:
    1. Outcome description similarity (70%)
    2. Final event similarity (30%)
    """
    # Outcome description similarity
    outcome_desc_sim = semantic_similarity(
        struct_a.get('outcome_description', ''),
        struct_b.get('outcome_description', '')
    )
    
    # Final/outcome events similarity
    chain_a = struct_a.get('events', [])
    chain_b = struct_b.get('events', [])
    
    def get_outcome_events(chain):
        outs = [ev for ev in chain if ev.get('is_outcome_event', False)]
        if not outs and chain:
            # Use last event if no outcome events marked
            outs = [chain[-1]]
        return outs
    
    outs_a = get_outcome_events(chain_a)
    outs_b = get_outcome_events(chain_b)
    
    # Compare outcome events
    if outs_a and outs_b:
        scores = []
        for ev_a in outs_a:
            for ev_b in outs_b:
                scores.append(event_similarity(ev_a, ev_b))
        final_event_sim = max(scores) if scores else 0.0
    else:
        final_event_sim = 0.0
    
    # Weighted combination
    total = 0.70 * outcome_desc_sim + 0.30 * final_event_sim
    return float(total)

print("Outcome similarity function ready.")

Outcome similarity function ready.


In [13]:
# Main narrative similarity function
def narrative_similarity(
    struct_a: Dict[str, Any],
    struct_b: Dict[str, Any],
    weights: Dict[str, float] = None
) -> Dict[str, float]:
    """
    Compute full narrative similarity between two story structures.
    
    Components:
    - theme_sim: Abstract theme similarity
    - avg_event_sim: Average event similarity (from optimal alignment)
    - order_score: Sequence preservation score
    - outcome_sim: Outcome similarity
    - combined: Weighted combination
    
    Default weights (can be tuned):
    - theme: 0.25
    - events: 0.35
    - order: 0.15
    - outcome: 0.25
    """
    if weights is None:
        weights = {
            'theme': 0.25,
            'events': 0.35,
            'order': 0.15,
            'outcome': 0.25
        }
    
    # Theme similarity
    theme_sim = semantic_similarity(
        struct_a.get('theme', ''),
        struct_b.get('theme', '')
    )
    
    # Event chain alignment and similarity
    chain_a = struct_a.get('events', [])
    chain_b = struct_b.get('events', [])
    aligned_pairs, avg_event_sim = align_event_chains_optimal(chain_a, chain_b)
    
    # Order preservation
    order_score = sequence_order_score(aligned_pairs)
    
    # Outcome similarity
    outcome_sim = outcome_similarity(struct_a, struct_b)
    
    # Combined score
    combined = (
        weights['theme'] * theme_sim +
        weights['events'] * avg_event_sim +
        weights['order'] * order_score +
        weights['outcome'] * outcome_sim
    )
    
    return {
        'theme_sim': float(theme_sim),
        'avg_event_sim': float(avg_event_sim),
        'order_score': float(order_score),
        'outcome_sim': float(outcome_sim),
        'combined': float(combined)
    }

print("Narrative similarity function ready.")

Narrative similarity function ready.


In [14]:
@dataclass
class NarrativeSimilarityResult:
    anchor_vs_a: Dict[str, float]
    anchor_vs_b: Dict[str, float]
    better: str     # "A" or "B"
    margin: float   # absolute difference

def compare_anchor_candidates(
    anchor: str,
    text_a: str,
    text_b: str,
    weights: Dict[str, float] = None,
    cache: Dict[str, Dict[str, Any]] = None
) -> NarrativeSimilarityResult:
    """
    Compare anchor vs A and anchor vs B.
    Determine which is more narratively similar.
    """
    if cache is None:
        cache = {}
    
    def get_structure(text: str) -> Dict[str, Any]:
        if text in cache:
            return cache[text]
        struct = extract_narrative_structure(text)
        cache[text] = struct
        return struct
    
    anchor_struct = get_structure(anchor)
    a_struct = get_structure(text_a)
    b_struct = get_structure(text_b)
    
    anchor_vs_a = narrative_similarity(anchor_struct, a_struct, weights)
    anchor_vs_b = narrative_similarity(anchor_struct, b_struct, weights)
    
    score_a = anchor_vs_a['combined']
    score_b = anchor_vs_b['combined']
    
    better = "A" if score_a >= score_b else "B"
    margin = abs(score_a - score_b)
    
    return NarrativeSimilarityResult(
        anchor_vs_a=anchor_vs_a,
        anchor_vs_b=anchor_vs_b,
        better=better,
        margin=margin
    )

print("Comparison function ready.")

Comparison function ready.


In [15]:
# Test on a single example
print("Testing on first row...\n")

row_idx = 0
anchor_story = df_dev.loc[row_idx, "anchor_text"]
story_a = df_dev.loc[row_idx, "text_a"]
story_b = df_dev.loc[row_idx, "text_b"]
label_a_is_closer = bool(df_dev.loc[row_idx, "text_a_is_closer"])

print("Gold label: A is closer? ->", label_a_is_closer)
print("\nExtracting narrative structures...")

res = compare_anchor_candidates(anchor_story, story_a, story_b)

print("\n" + "="*60)
print("RESULTS")
print("="*60)
print(f"\nAnchor vs A:")
for k, v in res.anchor_vs_a.items():
    print(f"  {k}: {v:.3f}")

print(f"\nAnchor vs B:")
for k, v in res.anchor_vs_b.items():
    print(f"  {k}: {v:.3f}")

print(f"\nPredicted better match: {res.better}")
print(f"Margin: {res.margin:.3f}")
print(f"\nPrediction: A_is_closer = {res.better == 'A'}")
print(f"Gold: A_is_closer = {label_a_is_closer}")
print(f"Correct: {(res.better == 'A') == label_a_is_closer}")

Testing on first row...

Gold label: A is closer? -> False

Extracting narrative structures...

RESULTS

Anchor vs A:
  theme_sim: 0.175
  avg_event_sim: 0.228
  order_score: 0.600
  outcome_sim: 0.034
  combined: 0.222

Anchor vs B:
  theme_sim: 0.251
  avg_event_sim: 0.193
  order_score: 0.200
  outcome_sim: 0.136
  combined: 0.194

Predicted better match: A
Margin: 0.028

Prediction: A_is_closer = True
Gold: A_is_closer = False
Correct: False


In [16]:
# Evaluation function
def evaluate_on_dataframe(
    df: pd.DataFrame,
    max_rows: int = 20,
    weights: Dict[str, float] = None,
    verbose: bool = True
) -> Dict[str, Any]:
    """
    Evaluate model on dev data.
    Returns accuracy and detailed results.
    """
    cache = {}
    preds = []
    gold = []
    detailed_results = []
    
    n = min(max_rows, len(df))
    
    for idx in range(n):
        row = df.iloc[idx]
        anchor = row["anchor_text"]
        a = row["text_a"]
        b = row["text_b"]
        label_a_is_closer = bool(row["text_a_is_closer"])
        gold.append(label_a_is_closer)
        
        res = compare_anchor_candidates(anchor, a, b, weights=weights, cache=cache)
        pred_is_a = (res.better == "A")
        preds.append(pred_is_a)
        
        correct = (pred_is_a == label_a_is_closer)
        
        if verbose:
            status = "✓" if correct else "✗"
            print(
                f"{idx+1}/{n}: {status} pred={res.better}, gold={'A' if label_a_is_closer else 'B'}, "
                f"margin={res.margin:.3f} | "
                f"A: theme={res.anchor_vs_a['theme_sim']:.2f} evt={res.anchor_vs_a['avg_event_sim']:.2f} "
                f"ord={res.anchor_vs_a['order_score']:.2f} out={res.anchor_vs_a['outcome_sim']:.2f} "
                f"→ {res.anchor_vs_a['combined']:.3f} | "
                f"B: theme={res.anchor_vs_b['theme_sim']:.2f} evt={res.anchor_vs_b['avg_event_sim']:.2f} "
                f"ord={res.anchor_vs_b['order_score']:.2f} out={res.anchor_vs_b['outcome_sim']:.2f} "
                f"→ {res.anchor_vs_b['combined']:.3f}"
            )
        
        detailed_results.append({
            'idx': idx,
            'predicted': res.better,
            'gold': 'A' if label_a_is_closer else 'B',
            'correct': correct,
            'margin': res.margin,
            'anchor_vs_a': res.anchor_vs_a,
            'anchor_vs_b': res.anchor_vs_b
        })
    
    preds = np.array(preds, dtype=bool)
    gold = np.array(gold, dtype=bool)
    acc = float((preds == gold).mean())
    
    print(f"\n{'='*60}")
    print(f"ACCURACY on {n} samples: {acc:.4f} ({int(acc*n)}/{n} correct)")
    print(f"{'='*60}")
    
    return {
        'accuracy': acc,
        'n_samples': n,
        'n_correct': int(acc * n),
        'detailed_results': detailed_results
    }

print("Evaluation function ready.")

Evaluation function ready.


In [27]:
# Quick sanity check - compare two dummy events
event_similarity = event_similarity
test_ev_a = {'action': 'lost item', 'actors': ['person'], 'object': 'wallet', 'result': 'worried'}
test_ev_b = {'action': 'misplaced object', 'actors': ['individual'], 'object': 'purse', 'result': 'concerned'}

sim = event_similarity(test_ev_a, test_ev_b)
print(f"Test similarity: {sim:.3f}")
# Should be higher than pure semantic (because 'lost' and 'misplaced' are semantically similar)

Test similarity: 0.539


In [29]:

weights_test3 = {
    'theme': 0.20,
    'events': 0.45,
    'order': 0.00,    # Ignore order entirely
    'outcome': 0.35
}

results_full = evaluate_on_dataframe(
    df_dev, 
    max_rows=len(df_dev),  
    weights=weights, 
    verbose=True 
)

1/200: ✓ pred=B, gold=B, margin=0.071 | A: theme=0.15 evt=0.21 ord=0.60 out=0.02 → 0.162 | B: theme=0.32 evt=0.22 ord=0.20 out=0.20 → 0.233
2/200: ✓ pred=A, gold=A, margin=0.010 | A: theme=0.39 evt=0.23 ord=0.60 out=0.25 → 0.287 | B: theme=0.23 evt=0.27 ord=0.33 out=0.32 → 0.277
3/200: ✓ pred=B, gold=B, margin=0.001 | A: theme=0.43 evt=0.21 ord=0.50 out=0.23 → 0.273 | B: theme=0.27 evt=0.26 ord=0.50 out=0.26 → 0.274
4/200: ✗ pred=A, gold=B, margin=0.006 | A: theme=0.55 evt=0.23 ord=0.60 out=0.23 → 0.310 | B: theme=0.31 evt=0.27 ord=0.40 out=0.34 → 0.304
5/200: ✓ pred=B, gold=B, margin=0.037 | A: theme=0.44 evt=0.31 ord=0.40 out=0.28 → 0.332 | B: theme=0.25 evt=0.34 ord=0.50 out=0.48 → 0.369
6/200: ✗ pred=A, gold=B, margin=0.100 | A: theme=0.33 evt=0.31 ord=0.60 out=0.39 → 0.353 | B: theme=0.47 evt=0.23 ord=0.50 out=0.10 → 0.253
7/200: ✗ pred=A, gold=B, margin=0.031 | A: theme=0.34 evt=0.24 ord=0.33 out=0.26 → 0.269 | B: theme=0.31 evt=0.26 ord=0.50 out=0.12 → 0.237
8/200: ✗ pred=A, gol

In [34]:
# Analyze the errors
errors = [r for r in results_full['detailed_results'] if not r['correct']]
correct = [r for r in results_full['detailed_results'] if r['correct']]

print(f"Errors: {len(errors)}/200")
print(f"Correct: {len(correct)}/200\n")

# Error margins
error_margins = [e['margin'] for e in errors]
correct_margins = [c['margin'] for c in correct]

print(f"Average margin when WRONG: {np.mean(error_margins):.3f}")
print(f"Average margin when CORRECT: {np.mean(correct_margins):.3f}")
print(f"Median margin when WRONG: {np.median(error_margins):.3f}")
print(f"Median margin when CORRECT: {np.median(correct_margins):.3f}\n")

# Component analysis for errors
print("Average component scores for ERRORS:")
error_theme_diff = []
error_event_diff = []
error_order_diff = []
error_outcome_diff = []

for e in errors:
    # Calculate which way the error went
    if e['predicted'] == 'A':
        # Predicted A but B was correct
        error_theme_diff.append(e['anchor_vs_a']['theme_sim'] - e['anchor_vs_b']['theme_sim'])
        error_event_diff.append(e['anchor_vs_a']['avg_event_sim'] - e['anchor_vs_b']['avg_event_sim'])
        error_order_diff.append(e['anchor_vs_a']['order_score'] - e['anchor_vs_b']['order_score'])
        error_outcome_diff.append(e['anchor_vs_a']['outcome_sim'] - e['anchor_vs_b']['outcome_sim'])
    else:
        # Predicted B but A was correct
        error_theme_diff.append(e['anchor_vs_b']['theme_sim'] - e['anchor_vs_a']['theme_sim'])
        error_event_diff.append(e['anchor_vs_b']['avg_event_sim'] - e['anchor_vs_a']['avg_event_sim'])
        error_order_diff.append(e['anchor_vs_b']['order_score'] - e['anchor_vs_a']['order_score'])
        error_outcome_diff.append(e['anchor_vs_b']['outcome_sim'] - e['anchor_vs_a']['outcome_sim'])

print(f"  Theme advantage (wrong pick): {np.mean(error_theme_diff):.3f}")
print(f"  Event advantage (wrong pick): {np.mean(error_event_diff):.3f}")
print(f"  Order advantage (wrong pick): {np.mean(error_order_diff):.3f}")
print(f"  Outcome advantage (wrong pick): {np.mean(error_outcome_diff):.3f}")

Errors: 86/200
Correct: 114/200

Average margin when WRONG: 0.043
Average margin when CORRECT: 0.043
Median margin when WRONG: 0.035
Median margin when CORRECT: 0.037

Average component scores for ERRORS:
  Theme advantage (wrong pick): 0.088
  Event advantage (wrong pick): 0.015
  Order advantage (wrong pick): 0.051
  Outcome advantage (wrong pick): 0.053


In [35]:
# Drastically reduce theme, maximize events
weights_optimal = {
    'theme': 0.05,     # Reduced from 0.10 
    'events': 0.65,    # Increased from 0.55 
    'order': 0.05,     # Reduced from 0.10 
    'outcome': 0.25    # Keep same 
}

print("Testing optimized weights based on error analysis...")
results_optimal = evaluate_on_dataframe(
    df_dev, 
    max_rows=20, 
    weights=weights_optimal, 
    verbose=False
)
print(f"\nAccuracy with optimized weights: {results_optimal['accuracy']:.4f}")

Testing optimized weights based on error analysis...

ACCURACY on 20 samples: 0.6500 (13/20 correct)

Accuracy with optimized weights: 0.6500
